# Misure rilevanti su un grafo diretto con NetworkX

Questo notebook mostra **solo le misure più utili** per analizzare un **grafo diretto** costruito da una *edge list* e indica **quando una misura è dispendiosa** per grafi grandi.

> **Nota:** per grafi molto grandi, alcune misure basate su cammini minimi o enumerazione di cicli possono diventare impraticabili.


## 0) Setup e costruzione del grafo

Assumiamo di avere una lista di archi orientati:

- `edges = [(u1, v1), (u2, v2), ...]`

Sostituisci/leggi i dati nel modo che preferisci (CSV, JSON, ecc.).

In [1]:
"""
import networkx as nx

# ESEMPIO: edge list giocattolo (sostituisci con i tuoi dati)
edges = [
    ("A","B"), ("B","C"), ("C","A"),
    ("C","D"), ("D","E"), ("E","D"),
    ("F","G")
]

G = nx.DiGraph()
G.add_edges_from(edges)

print("Nodi:", G.number_of_nodes())
print("Archi:", G.number_of_edges())
"""
import pandas as pd
import networkx as nx
import os
import json
from tqdm import tqdm

# Path del file edges (salvato per non ricalcolare)
EDGES_FILE = '../../material/tgdataset_edges.csv.gz'
EXTRACTED_DIR = '../../material/TGDataset_extracted/public_db'

def extract_all_edges():
    """Estrae tutti gli archi (forwarded_from_id → channel_id) da TGDataset."""
    edges = []
    files_processed = 0
    files_failed = 0
    
    # Trova tutte le cartelle folder_X
    folders = [f for f in os.listdir(EXTRACTED_DIR) if f.startswith('folder_')]
    print(f"Trovate {len(folders)} cartelle")
    
    for folder in tqdm(folders, desc="Cartelle"):
        folder_path = f"{EXTRACTED_DIR}/{folder}"
        json_files = [f for f in os.listdir(folder_path) if f.endswith('.json')]
        
        for json_file in json_files:
            file_path = f"{folder_path}/{json_file}"
            try:
                with open(file_path, 'r') as f:
                    data = json.load(f)
                
                files_processed += 1
                
                # Ogni chiave è un channel_id
                for channel_id, channel_data in data.items():
                    text_messages = channel_data.get('text_messages', {})
                    
                    for msg_id, msg in text_messages.items():
                        fwd_from = msg.get('forwarded_from_id')
                        if fwd_from is not None:
                            edges.append({
                                'source': int(fwd_from),
                                'target': int(channel_id)
                            })
            except Exception as e:
                files_failed += 1
    
    print(f"\n=== ESTRAZIONE COMPLETATA ===")
    print(f"  File processati: {files_processed:,}")
    print(f"  File falliti: {files_failed:,}")
    print(f"  Archi estratti: {len(edges):,}")
    
    return pd.DataFrame(edges)

# Carica o crea il file edges
if os.path.exists(EDGES_FILE):
    print(f"Caricamento edges da {EDGES_FILE}...")
    edges_df = pd.read_csv(EDGES_FILE, compression='gzip')
    print(f"Caricati {len(edges_df):,} archi")
else:
    print("Estrazione archi da TGDataset (può richiedere 1-2 ore)...")
    edges_df = extract_all_edges()
    edges_df = edges_df.drop_duplicates()
    edges_df.to_csv(EDGES_FILE, index=False, compression='gzip')
    print(f"Salvati {len(edges_df):,} archi unici in {EDGES_FILE}")

# Crea il grafo
G = nx.DiGraph()
G.add_edges_from(zip(edges_df['source'], edges_df['target']))

# ======================
# CONTROLLI DI VALIDITÀ
# ======================
print(f"\n{'='*50}")
print(f" CONTROLLI DI VALIDITÀ")
print(f"{'='*50}")

# 1. Check dimensioni minime
assert len(edges_df) > 1000, f"ERRORE: Troppi pochi archi ({len(edges_df)})"
print(f"✅ Archi totali: {len(edges_df):,}")

# 2. Check nodi
assert G.number_of_nodes() > 100, f"ERRORE: Troppi pochi nodi ({G.number_of_nodes()})"
print(f"✅ Nodi totali: {G.number_of_nodes():,}")

# 3. Check no valori nulli
null_count = edges_df.isnull().sum().sum()
assert null_count == 0, f"ERRORE: Trovati {null_count} valori nulli"
print(f"✅ Nessun valore nullo")

# 4. Check tipi corretti
assert edges_df['source'].dtype in ['int64', 'int32', 'float64'], "ERRORE: source non numerico"
assert edges_df['target'].dtype in ['int64', 'int32', 'float64'], "ERRORE: target non numerico"
print(f"✅ Tipi corretti (source: {edges_df['source'].dtype}, target: {edges_df['target'].dtype})")

# 5. Statistiche base
print(f"\n{'='*50}")
print(f" STATISTICHE GRAFO")
print(f"{'='*50}")
print(f"  Nodi: {G.number_of_nodes():,}")
print(f"  Archi: {G.number_of_edges():,}")
print(f"  Densità: {nx.density(G):.6f}")
print(f"  Self-loops: {nx.number_of_selfloops(G):,}")

# 6. Componenti connesse
n_wcc = nx.number_weakly_connected_components(G)
largest_wcc = max(nx.weakly_connected_components(G), key=len)
print(f"  Componenti (WCC): {n_wcc:,}")
print(f"  Largest WCC: {len(largest_wcc):,} nodi ({100*len(largest_wcc)/G.number_of_nodes():.1f}%)")

# 7. Grado medio
avg_in = sum(dict(G.in_degree()).values()) / G.number_of_nodes()
avg_out = sum(dict(G.out_degree()).values()) / G.number_of_nodes()
print(f"  In-degree medio: {avg_in:.2f}")
print(f"  Out-degree medio: {avg_out:.2f}")

print(f"\n✅ GRAFO PRONTO PER L'ANALISI")

Estrazione archi da TGDataset (può richiedere 1-2 ore)...
Trovate 4 cartelle


Cartelle: 100%|██████████| 4/4 [1:40:37<00:00, 1509.32s/it]



=== ESTRAZIONE COMPLETATA ===
  File processati: 121
  File falliti: 0
  Archi estratti: 52,979,776
Salvati 6,955,167 archi unici in ../../material/tgdataset_edges.csv.gz

 CONTROLLI DI VALIDITÀ
✅ Archi totali: 6,955,167
✅ Nodi totali: 1,117,812
✅ Nessun valore nullo
✅ Tipi corretti (source: int64, target: int64)

 STATISTICHE GRAFO
  Nodi: 1,117,812
  Archi: 6,955,167
  Densità: 0.000006
  Self-loops: 49,454
  Componenti (WCC): 530
  Largest WCC: 1,116,361 nodi (99.9%)
  In-degree medio: 6.22
  Out-degree medio: 6.22

✅ GRAFO PRONTO PER L'ANALISI


## 1) Dimensione e densità (sempre fattibili)

- **Numero di nodi**: `G.number_of_nodes()`
- **Numero di archi**: `G.number_of_edges()`
- **Densità**: `nx.density(G)`

**Costo:** ~O(1)

In [2]:
n = G.number_of_nodes()
m = G.number_of_edges()
dens = nx.density(G)

n, m, dens

(1117812, 6955167, 5.566348786247016e-06)

## 2) In-degree e out-degree (fondamentali)

- **In-degree**: numero di archi entranti
- **Out-degree**: numero di archi uscenti

**Costo:** ~O(|E|) (scala bene anche su grafi grandi)

In [3]:
in_deg  = dict(G.in_degree())
out_deg = dict(G.out_degree())

# Top-10 per in-degree / out-degree
top_in  = sorted(in_deg.items(),  key=lambda x: x[1], reverse=True)[:10]
top_out = sorted(out_deg.items(), key=lambda x: x[1], reverse=True)[:10]

top_in, top_out

([(1303782697, 2970),
  (1145761925, 2455),
  (1102334403, 2287),
  (1225656516, 2020),
  (1004035975, 1957),
  (1133019113, 1846),
  (1241034426, 1761),
  (1334602589, 1755),
  (1335904171, 1730),
  (1440393212, 1604)],
 [(1394050290, 6172),
  (1117628569, 5740),
  (1101170442, 5601),
  (200389711, 4808),
  (1179270258, 4379),
  (1069772626, 4269),
  (1096463945, 4194),
  (1479163323, 3761),
  (1378813139, 3759),
  (1036362176, 3688)])

### Centralità di grado (normalizzata)

- `nx.in_degree_centrality(G)`
- `nx.out_degree_centrality(G)`

**Costo:** ~O(|E|)

In [4]:
in_deg_cent  = nx.in_degree_centrality(G)
out_deg_cent = nx.out_degree_centrality(G)

# Top-10
sorted(in_deg_cent.items(), key=lambda x: x[1], reverse=True)[:10], sorted(out_deg_cent.items(), key=lambda x: x[1], reverse=True)[:10]

([(1303782697, 0.0026569786842319497),
  (1145761925, 0.0021962567911748944),
  (1102334403, 0.0020459630474203598),
  (1225656516, 0.0018071033475247605),
  (1004035975, 0.00175074319361681),
  (1133019113, 0.0016514419700647068),
  (1241034426, 0.001575400492569853),
  (1334602589, 0.0015700328588643339),
  (1335904171, 0.001547667718424671),
  (1440393212, 0.00143494741060877)],
 [(1394050290, 0.005521505871743971),
  (1117628569, 0.005135036244946597),
  (1101170442, 0.005010686064102071),
  (200389711, 0.004301263809355965),
  (1179270258, 0.00391747799941135),
  (1069772626, 0.003819071381476833),
  (1096463945, 0.003751975960157844),
  (1479163323, 0.0033646117277428833),
  (1378813139, 0.00336282251650771),
  (1036362176, 0.0032993055176590677)])

## 3) Componenti connesse (molto importanti)

Nei grafi diretti si usa spesso:

- **SCC** (*Strongly Connected Components*): nodi mutuamente raggiungibili seguendo le direzioni.
- **WCC** (*Weakly Connected Components*): connettività ignorando l'orientamento.

**Costo:** ~O(|V| + |E|) (fattibile anche su grafi grandi)

In [5]:
scc = list(nx.strongly_connected_components(G))
wcc = list(nx.weakly_connected_components(G))

num_scc = len(scc)
num_wcc = len(wcc)

largest_scc = max(scc, key=len)
largest_wcc = max(wcc, key=len)

num_scc, len(largest_scc), num_wcc, len(largest_wcc)

(1031451, 85361, 530, 1116361)

## 4) PageRank (chiave nei grafi diretti)

- `nx.pagerank(G)` calcola un punteggio di importanza dei nodi.
- È un metodo **iterativo**.

**Costo:** ~O(k·|E|) dove *k* è il numero di iterazioni.  
Su grafi grandi può essere costoso: controlla `max_iter` e `tol`.

In [6]:
pr = nx.pagerank(G, max_iter=100, tol=1e-6)
top_pr = sorted(pr.items(), key=lambda x: x[1], reverse=True)[:10]
top_pr

[(1004035975, 0.0013160223339057843),
 (1064404455, 0.0011845110571447816),
 (1433940064, 0.0008196707351091837),
 (1324999204, 0.000813512565125187),
 (1448885855, 0.0007994770522526013),
 (1124923953, 0.0006804512155754678),
 (1102334403, 0.0006632936318994503),
 (1046019295, 0.0005975956699730039),
 (1145761925, 0.0005905429182468668),
 (1133019113, 0.00055890268436765)]

## 5) HITS (hub e authority)

- `nx.hits(G)` restituisce due dizionari: **hubs** e **authorities**.
- Anche questo è **iterativo**.

**Costo:** ~O(k·|E|)  
Su grafi grandi può essere costoso e talvolta più instabile di PageRank.

In [7]:
hubs, auth = nx.hits(G, max_iter=1000, tol=1e-8)

top_hubs = sorted(hubs.items(), key=lambda x: x[1], reverse=True)[:10]
top_auth = sorted(auth.items(), key=lambda x: x[1], reverse=True)[:10]

top_hubs, top_auth

([(1296087183, 0.001644106321727443),
  (1353291112, 0.0016438643012409537),
  (1489529243, 0.0016432882456483432),
  (1343638843, 0.0016423058512385542),
  (1265080114, 0.0016423003485455797),
  (1450000704, 0.0016421384204552286),
  (1254793878, 0.0016407568923847175),
  (1303844417, 0.0016403315997253957),
  (1371199592, 0.0016400089827523724),
  (1424937539, 0.001639691173583781)],
 [(1319837199, 0.0014660938636441932),
  (1212645245, 0.001464074926052309),
  (1318275860, 0.001461853319240805),
  (1461240728, 0.001459738401871254),
  (1347229123, 0.0014581495257246736),
  (1215834055, 0.0014572565826898249),
  (1361322669, 0.0014535383803713647),
  (1263113710, 0.0014521381295638325),
  (1444621361, 0.0014520848716576115),
  (1277748571, 0.001451523847166496)])

## 6) Betweenness centrality (da usare con cautela)

- `nx.betweenness_centrality(G)` misura quanto un nodo fa da “ponte” tra percorsi.

**Costo:** ~O(|V|·|E|)  
👉 **Molto dispendioso**: sconsigliato su grafi grandi.

Suggerimento pratico: su grafi grandi considera un'**approssimazione** scegliendo un campione di sorgenti con `k=`.

In [ ]:
# ESEMPIO: su grafi grandi usare una versione approssimata:
# approx_bc = nx.betweenness_centrality(G, k=100, seed=42)

bc = nx.betweenness_centrality(G)  # ok solo su grafi piccoli/medi
sorted(bc.items(), key=lambda x: x[1], reverse=True)[:10]

## 7) Closeness centrality (attenzione)

- `nx.closeness_centrality(G)` usa molte distanze minime.

**Costo:** ~O(|V|·(|V|+|E|))  
👉 **Dispendioso** su grafi grandi e poco robusto se il grafo non è ben connesso.

In [ ]:
cc = nx.closeness_centrality(G)
sorted(cc.items(), key=lambda x: x[1], reverse=True)[:10]

## 8) Distanze globali / all-pairs shortest paths (solo su sottografi)

- `nx.all_pairs_shortest_path_length(G)` calcola le distanze minime **da ogni nodo verso tutti i raggiungibili**.

**Costo:** ~O(|V|·(|V|+|E|))  
👉 **Impraticabile** su grafi grandi.

Consiglio: applicare la misura **solo alla SCC più grande**.

In [ ]:
# Costruiamo il sottografo della SCC più grande (tipica scelta)
H = G.subgraph(largest_scc).copy()

# ATTENZIONE: anche su H può essere costoso se H è grande
all_pairs = dict(nx.all_pairs_shortest_path_length(H))

# Esempio: media delle distanze (incluse le 0 su se stessi)
dists = []
for u, dist_dict in all_pairs.items():
    dists.extend(dist_dict.values())

avg_dist = sum(dists) / len(dists) if dists else None
len(H), avg_dist

## 9) Cicli e DAG

- `nx.is_directed_acyclic_graph(G)` verifica se il grafo è un **DAG** (nessun ciclo diretto).  
  **Costo:** ~O(|V|+|E|)

- `nx.simple_cycles(G)` elenca i cicli semplici.  
  **Costo:** potenzialmente **esponenziale** 👉 evitare su grafi grandi.

In [ ]:
is_dag = nx.is_directed_acyclic_graph(G)
is_dag

In [ ]:
# ATTENZIONE: può esplodere su grafi grandi
cycles = list(nx.simple_cycles(G))
cycles

## 10) Reciprocità (utile e leggera)

Misura quanto gli archi sono spesso **bidirezionali**.

- `nx.reciprocity(G)` -> valore globale (0..1 circa)

**Costo:** ~O(|E|) (molto fattibile)

In [ ]:
nx.reciprocity(G)

## Riepilogo pratico (cosa usare su grafi grandi)

✅ **Sempre ok**: nodi/archi/densità, in/out-degree, SCC/WCC, reciprocità  
⚠️ **Ok con attenzione**: PageRank, HITS (iterativi)  
❌ **Spesso da evitare**: betweenness, closeness, all-pairs shortest paths, enumerazione cicli
